# Current-best error diagnosis before V013 screening

Descriptive A validation diagnostics only. This validation has been used repeatedly for research; these examples are not independent evidence. Pressure labels below are for inspection only and never become model inputs.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
root = Path.cwd()
out = root / "outputs/controlled_experiments_v013/diagnosis"
out.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(root / "data/train.csv", usecols=["id", "breath_id", "R", "C", "u_out", "pressure"], dtype={"id":"int64", "breath_id":"int32", "R":"int8", "C":"int8", "u_out":"int8", "pressure":"float32"})
bids = train.breath_id.to_numpy()[::80]
with np.load(root / "outputs/improvements_v005/sequence/split_breath_ids.npz") as z:
    val = np.sort(pd.Index(bids).get_indexer(z["A_validation"]))
y = train.pressure.to_numpy().reshape(-1,80)[val]
mask = train.u_out.to_numpy().reshape(-1,80)[val] == 0
ids = train.id.to_numpy().reshape(-1,80)[val][mask]
rowpath = root / "outputs/diversity_screen_v012/A_row_ids.npy"
assert np.array_equal(ids, np.load(rowpath))
predpath = root / "outputs/diversity_screen_v012/A_deep_half_compact_curve_memory_offset_predictions.npy"
pred = np.load(predpath)
err = np.abs(pred - y[mask].astype(np.float64))
rc = train[["R","C"]].to_numpy()[::80][val]
points_rc = np.repeat(rc,80,axis=0)[mask.ravel()]
step = np.broadcast_to(np.arange(80),mask.shape)[mask]
table = pd.DataFrame(dict(R=points_rc[:,0], C=points_rc[:,1], step=step, error=err))
rc_table = table.groupby(["R","C"]).agg(points=("error","size"),MAE=("error","mean"),total_error=("error","sum")).reset_index()
rc_table["error_share"] = rc_table.total_error / err.sum()
rc_table = rc_table.sort_values("total_error",ascending=False)
step_table = table.groupby("step").agg(points=("error","size"),MAE=("error","mean"),total_error=("error","sum")).reset_index()
rc_table.to_csv(out / "reference_A_by_RC.csv",index=False)
step_table.to_csv(out / "reference_A_by_step.csv",index=False)
counts = mask.sum(1)
sums = np.add.reduceat(err,np.r_[0,np.cumsum(counts)[:-1]])
breath_mae = sums / counts
worst = np.argsort(-breath_mae)
top10 = worst[:int(np.ceil(.1*len(worst))) ]
report = dict(scope="Descriptive diagnostics of existing A predictions; no new model fits or new candidate evaluation",mean_MAE=float(err.mean()),median_absolute_error=float(np.median(err)),top_10_percent_breaths_error_share=float(sums[top10].sum()/err.sum()),RC_contributions=rc_table.to_dict("records"),prediction_file_sha256=hashlib.sha256(predpath.read_bytes()).hexdigest(),worst_breath_ids=[int(bids[val[k]]) for k in worst[:4]])
(out / "summary.json").write_text(json.dumps(report,indent=2)+"\n")
print(rc_table.to_string(index=False))
print("Worst 10% of breaths contribute {:.1%} of total absolute error".format(report["top_10_percent_breaths_error_share"]))
fullpred = np.full(y.shape,np.nan)
fullpred[mask] = pred
fig,axes = plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
for ax,k in zip(axes.ravel(),worst[:4]):
    ix=np.flatnonzero(mask[k])
    ax.plot(ix,y[k,ix],label="Actual",linewidth=1.5)
    ax.plot(ix,fullpred[k,ix],label="Current ensemble",linewidth=1.3)
    ax.set_title(f"Breath {bids[val[k]]}: R={rc[k,0]}, C={rc[k,1]}, MAE={breath_mae[k]:.3f}")
    ax.set_xlabel("Position within breath");ax.set_ylabel("Pressure");ax.legend()
fig.savefig(out / "difficult_breaths.png",dpi=140)
plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
axes[0].bar([f"R={r.R}, C={r.C}" for r in rc_table.itertuples()],rc_table.error_share)
axes[0].tick_params(axis="x",rotation=70);axes[0].set_ylabel("Share of total absolute error")
axes[1].bar(step_table.step,step_table.total_error);axes[1].set_xlabel("Position within breath");axes[1].set_ylabel("Total absolute error")
fig.savefig(out / "error_contributions.png",dpi=140)
plt.close(fig)


 R  C  points      MAE  total_error  error_share
50 10   37444 0.196247  7348.262213     0.186817
50 20   23137 0.278842  6451.561531     0.164020
50 50   22739 0.264902  6023.599745     0.153140
 5 10   22461 0.183770  4127.660537     0.104939
20 50   21893 0.179482  3929.401531     0.099898
20 10   16589 0.187350  3107.945955     0.079014
 5 50   22061 0.128715  2839.579051     0.072191
 5 20   23070 0.119439  2755.451333     0.070053
20 20   16704 0.164664  2750.553524     0.069928
Worst 10% of breaths contribute 40.4% of total absolute error


![Error contributions](../outputs/controlled_experiments_v013/diagnosis/error_contributions.png)

![Difficult breaths](../outputs/controlled_experiments_v013/diagnosis/difficult_breaths.png)